In [1]:
import os
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import mlflow
import mlflow.xgboost

/Users/abhishek/Documents/Projects/credit-score-mlops/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Credit_Risk_Scoring1")

2026/10/16 12:28:42 INFO mlflow.tracking.fluent: Experiment with name 'Credit_Risk_Scoring1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1792133922224, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792133922224, lifecycle_stage='active', name='Credit_Risk_Scoring1', tags={}, trace_location=None, workspace='default'>

In [4]:
with mlflow.start_run() as run:
    print("Loading DVC tracked dataset")
    df = pd.read_csv("data/raw/credit_card.csv")
    df.columns = df.columns.str.strip()

    target_col = "Class"

    x = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42, stratify=y)


    # hyper parameter 
    params = {
        "max_depth": 5,
        "learning_rate":0.1,
        "n_estimators": 100,
        "eval_metrics": "logloss"
    }

    mlflow.xgboost.autolog() # automatically logs metrics parameters and model artifacts 

    print("XGB Classifier Training")
    model = xgb.XGBClassifier(**params)
    model.fit(X_train,y_train)


    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print("model", model.predict_proba)

    cal_f1 = f1_score(y_test,preds, average='weighted')
    roc_auc = roc_auc_score(y_test,probs)

    mlflow.log_metric("custom f1 score ", cal_f1)
    
    mlflow.log_metric("Custom roc auc score", roc_auc)

    git_commit = os.popen("git rev-parse HEAD").read().strip()
    mlflow.set_tag("data_git_commit", git_commit)

    print(f"Training complete! Logged to MLflow Run ID: {run.info.run_id}")

Loading DVC tracked dataset
XGB Classifier Training


/Users/abhishek/Documents/Projects/credit-score-mlops/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [12:38:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/10/16 12:38:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/16 12:38:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


model <bound method XGBClassifier.predict_proba of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, eval_metrics='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None, ...)>
Training complete! Logged to MLflow Run ID: c765289fa7e94420be43593d29606a1c
🏃 View run welcoming-eel-435 at: http://127.0.0.1:5000/#/experiments/1/runs/c765289fa7e94420be43593d29606a1c
🧪 View 

In [ ]:
# Extracting model 
mlflow.set_tracking_uri("http://127.0.0.1:5000")

os.makedirs("model",exist_ok=True)
RUN_ID = "c765289fa7e94420be43593d29606a1c"
artifact_path = mlflow.artifacts.download_artifacts(run_id=RUN_ID,artifact_path="model")

In [12]:
print(f"Model downloaded from MLflow to: {artifact_path}")
import shutil

Model downloaded from MLflow to: /var/folders/m5/jj_p1yf95jz65s2sx12v_flh0000gn/T/tmpohy7m_1s/model


In [14]:
if os.path.exists("model"):
    shutil.rmtree("model")
shutil.copytree(artifact_path, "model")

'model'